# Chapter 17 · When “yes” is rare
**Deck 3 · Live demo.** Judge a dropout model properly: four boxes, recall and precision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
100 students, 3 drop out. A model that **never** flags anyone: what accuracy? How many dropouts does it catch?

In [ ]:
my_prediction = {'accuracy': 0.97, 'caught': 0}

## 2 · The 100 students
Each has a risk score from a model (the same scores as the slide calculator).

In [ ]:
drop = np.array([0.9, 0.7, 0.3])                                   # 3 dropouts
stay = np.r_[[0.8, 0.65, 0.55], 0.02 + 0.43 * np.arange(94) / 93]   # 97 who stay
scores = np.r_[drop, stay]
truth = np.r_[np.ones(3), np.zeros(97)]                            # 1 = dropped out
print(len(scores), 'students ·', int(truth.sum()), 'dropouts')

## 3 · Four boxes

In [ ]:
def boxes(threshold):
    flag = scores >= threshold
    tp = int(np.sum(flag & (truth == 1))); fp = int(np.sum(flag & (truth == 0)))
    fn = int(np.sum(~flag & (truth == 1))); tn = int(np.sum(~flag & (truth == 0)))
    return tp, fp, fn, tn

tp, fp, fn, tn = boxes(0.5)
print(f'caught {tp} · false alarms {fp} · missed {fn} · correctly left alone {tn}')

## 4 · Scores

In [ ]:
accuracy = (tp + tn) / 100
recall = tp / (tp + fn)
precision = tp / (tp + fp)
lazy_accuracy = 97 / 100
print(f'accuracy {accuracy:.0%} (lazy {lazy_accuracy:.0%}) · recall {recall:.0%} · precision {precision:.0%}')

## 5 · PROOF

In [ ]:
assert (tp, fp, fn, tn) == (2, 3, 1, 94), 'the four boxes on the slide'
assert accuracy == 0.96 < lazy_accuracy == 0.97, 'the model is LESS accurate than doing nothing…'
assert tp == 2 and my_prediction['caught'] == 0, '…yet it catches 2 of 3 dropouts'
assert round(recall, 2) == 0.67 and precision == 0.4, 'recall 67% · precision 40%'
assert boxes(0.25)[:3] == (3, 47, 0), 'threshold 0.25: all caught, 50 calls'
print('✓ proof: 96% < 97%, yet 2 of 3 caught · recall 67% · precision 40%')

## 6 · Sweep the threshold

In [ ]:
ts = np.round(np.arange(0.05, 0.96, 0.05), 2)
rec = [boxes(t)[0] / 3 for t in ts]
prec = [boxes(t)[0] / max(1, boxes(t)[0] + boxes(t)[1]) for t in ts]
plt.plot(ts, rec, 'o-', label='recall (dropouts caught)')
plt.plot(ts, prec, 's-', label='precision (calls needed)')
plt.xlabel('threshold'); plt.legend(); plt.show()

## 6b · The trade-off table (as on the slide)

In [ ]:
print(f"{'risk ≥':>7} {'calls':>6} {'caught':>7} {'missed':>7} {'accuracy':>9} {'recall':>7} {'precision':>10}")
rows = {}
for t in (0.3, 0.5, 0.75, 0.85):
    tp, fp, fn, tn = boxes(t)
    rows[t] = (tp + fp, tp, fn, tp + tn, round(100 * tp / 3), round(100 * tp / (tp + fp)))
    print(f'{t:>7} {tp + fp:>6} {tp:>7} {fn:>7} {tp + tn:>8}% {rows[t][4]:>6}% {rows[t][5]:>9}%')
assert rows[0.3] == (39, 3, 0, 64, 100, 8) and rows[0.5] == (5, 2, 1, 96, 67, 40)
assert rows[0.75] == (2, 1, 2, 97, 33, 50) and rows[0.85] == (1, 1, 2, 98, 33, 100)
print('✓ accuracy is highest (98%) when we catch only 1 of 3')

## 7 · Your turn
Pick the threshold you would defend to the head teacher, and explain the trade-off in one sentence.

In [ ]:
# TODO

## Recap
When “yes” is rare, accuracy lies. Count the four boxes; report recall and precision; choose the threshold by what each mistake costs.